# 04e — EXP_05: Multi-Hop RAG (3-hop iterative dense retrieval)

**Architecture:** iterative dense retrieval with sub-query generation. Each question runs ≤3 retrieval rounds; rounds 2–3 use a sub-query generated by Groq from (original question + accumulated chunks); chunks are deduplicated across rounds; up to **15 final chunks** passed to the answerer (vs 5 for EXP_02–04). **Generator (answer + sub-queries):** `llama-3.3-70b-versatile` via Groq · T=0 · max_tokens=700 (answer) / 80 (sub-query).

Same three-stage gating pattern. **Note**: per-question Groq cost is ~3× higher than EXP_02 (1 answer + 2 sub-queries vs 1 answer), and `k=15` retrieved chunks means longer prompts → longer answer-generation latency. Wall-time projection on test_1273: **~30–45 min** (vs EXP_02's 12 min).

**The EXP_01 / EXP_02 / EXP_03 / EXP_04 baselines to compare against** (canonical test_1273):
- EXP_01 No-RAG: `Acuuracy = 0.7738`
- EXP_02 Naive Dense: `0.7573`
- EXP_03 Sparse BM25: `0.7581`
- EXP_04 Hybrid: (pending)

**Falsifiable hypothesis for EXP_05** (anchored in [`docs/output_notes/04b_exp02_output.md` §3.5](../docs/output_notes/04b_exp02_output.md)): *Multi-hop architecture achieves Faithfulness > 0 on the 13 multi-hop golden rows where Naive scored 0.000. Falsified if Multi-Hop Faithfulness on `requires_multihop=yes` rows ≤ 0.05.*

Plus a secondary hypothesis: *Multi-Hop should clear EXP_02's Acuuracy on test_1273 (≥ 0.7573) — if iterative retrieval doesn't even match Naive, the 3× compute cost isn't earning its keep.*

---

Stages:

| Stage | Surface | Wall time | Cost | Gate |
|---|---|---|---|---|
| **A** Smoke | 50 stratified rows | ~3–5 min | $0 | always on |
| **B** Golden | 234 accepted golden rows | ~5–8 min | $0 | `RUN_GOLDEN = True` |
| **C** Test | 1,273 test-split rows | ~30–45 min | $0 | `RUN_FULL = False` until A and B clean |

## 1. Setup

In [ ]:
import json
import logging
import os
import sys
from pathlib import Path

os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb").setLevel(logging.WARNING)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.data.embedder import best_device, load_bge
from src.data.indices import load_chroma
from src.data.loaders import load_golden, load_medqa_4opt
from src.eval.runner import run_experiment
from src.retrieval.multi_hop import MultiHopRetriever

print("GROQ_API_KEY present:", "\u2713" if os.environ.get("GROQ_API_KEY") else "\u2717")
print("Repo root:", REPO_ROOT)
print("Device   :", best_device())

## 2. Build the retriever (BGE-large + ChromaDB + Groq sub-query)

In [ ]:
CHROMA_DIR = REPO_ROOT / "data" / "indices" / "chroma_textbooks"
MAX_HOPS = 3   # plan.md §0 #7
PER_HOP_K = 5  # plan.md §6

chroma_coll = load_chroma(CHROMA_DIR)
embedder = load_bge(device=best_device())
RETRIEVER = MultiHopRetriever(
    embedder, chroma_coll,
    max_hops=MAX_HOPS, per_hop_k=PER_HOP_K,
)

print(f"ChromaDB count : {chroma_coll.count():,}")
print(f"max_hops       : {MAX_HOPS}")
print(f"per_hop_k      : {PER_HOP_K}")
print(f"max final k    : {MAX_HOPS * PER_HOP_K}")

# Sanity check on the CAP query
_test_chunks = RETRIEVER.retrieve(
    "What is the first-line treatment for community-acquired pneumonia?", k=15
)
print(f"\nSanity query (CAP first-line treatment) — returned {len(_test_chunks)} chunks across hops:")
for i, c in enumerate(_test_chunks[:5]):
    print(f"  [{c.score:.4f}] {c.chunk_id} ({c.book_name}): {c.text[:90].replace(chr(10), ' ')}\u2026")
print(f"  ... (+{len(_test_chunks) - 5} more)")

## 3. Configuration

In [ ]:
EXPERIMENT_ID = "EXP_05_MULTI_HOP_RAG"
RESULTS_DIR = REPO_ROOT / "results"
TOP_K = MAX_HOPS * PER_HOP_K  # = 15 — pass all accumulated chunks to the answerer

SMOKE_N = 50
SMOKE_SEED = 42

RUN_SMOKE = True
RUN_GOLDEN = True
RUN_FULL = False  # Stage C · ~30–45 min Groq on test split (1,273)

## 4. Stage A — Smoke (50 stratified rows)

In [ ]:
if RUN_SMOKE:
    medqa = load_medqa_4opt()
    smoke = (
        medqa.groupby("meta_info", group_keys=False)
        .apply(lambda g: g.sample(n=max(1, round(SMOKE_N * len(g) / len(medqa))), random_state=SMOKE_SEED))
        .reset_index(drop=True)
    )
    smoke = smoke.head(SMOKE_N) if len(smoke) >= SMOKE_N else medqa.sample(n=SMOKE_N, random_state=SMOKE_SEED).reset_index(drop=True)

    print(f"Smoke surface: {len(smoke)} rows | meta_info mix: {dict(smoke['meta_info'].value_counts())}")

    smoke_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=smoke,
        output_dir=RESULTS_DIR / "exp_05_multi_hop_rag__smoke_50",
        experiment_id=EXPERIMENT_ID,
        dataset_label="smoke_50",
        k=TOP_K,
    )
    print(json.dumps(smoke_summary, indent=2))
else:
    print("RUN_SMOKE = False — skipping Stage A")

**Acceptance gate (Stage A):** `n_questions == 50`; `Acuuracy ≥ 0.70` (multi-hop is more uncertain than single-shot — passing any contention with longer prompts is a sanity-check threshold); `mean_latency_s` 1.0–2.5 s (the longer prompt slows generation; sub-query generation also adds time); 0 null `pred_letter`s; `retrieval.jsonl` rows have ~10–15 chunk IDs (varies due to early-stop on no-progress).

## 5. Stage B — Golden 234

In [ ]:
if RUN_GOLDEN:
    golden = load_golden()
    print(f"Golden surface: {len(golden)} accepted rows")

    golden_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=golden,
        output_dir=RESULTS_DIR / "exp_05_multi_hop_rag__golden_234",
        experiment_id=EXPERIMENT_ID,
        dataset_label="golden_234",
        k=TOP_K,
    )
    print(json.dumps(golden_summary, indent=2))
else:
    print("RUN_GOLDEN = False — skipping Stage B")

## 6. Stage C — Test split (1,273 rows · gated · ~30–45 min)

Flip `RUN_FULL = True` only after Stages A and B look right.

In [ ]:
if RUN_FULL:
    medqa = load_medqa_4opt()
    medqa = medqa[medqa["split"] == "test"].reset_index(drop=True)
    print(f"Test-split surface: {len(medqa)} rows")

    full_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=medqa,
        output_dir=RESULTS_DIR / "exp_05_multi_hop_rag__test_1273",
        experiment_id=EXPERIMENT_ID,
        dataset_label="test_1273",
        k=TOP_K,
    )
    print(json.dumps(full_summary, indent=2))
else:
    print("RUN_FULL = False — skipping Stage C")

## 7. Inspect — does Multi-Hop earn its 3× compute cost?

In [ ]:
import pandas as pd

for label in ("smoke_50", "golden_234", "test_1273"):
    p = RESULTS_DIR / f"exp_05_multi_hop_rag__{label}" / "predictions.jsonl"
    if not p.exists():
        continue
    preds = pd.DataFrame(json.loads(line) for line in p.read_text().splitlines())
    print(f"\n=== {label}  (n={len(preds)}) ===")
    print(f"  EXP_05 Multi-Hop Acuuracy : {preds['is_correct'].mean():.4f}")
    print(f"  mean latency              : {preds['latency_s'].mean():.3f} s")

    if label == "test_1273":
        e1 = json.loads((REPO_ROOT / 'results' / 'exp_01_base_llm__test_1273' / 'summary.json').read_text())
        e2 = json.loads((REPO_ROOT / 'results' / 'exp_02_naive_rag__test_1273' / 'summary.json').read_text())
        e3 = json.loads((REPO_ROOT / 'results' / 'exp_03_sparse_rag__test_1273' / 'summary.json').read_text())
        e5_acc = preds['is_correct'].mean()
        print()
        print(f'  Hypothesis check — Multi-Hop Acuuracy ≥ EXP_02 (0.7573)?  {"\u2713 SUPPORTED" if e5_acc >= 0.7573 else "\u2717 FALSIFIED"}')
        print(f'    EXP_01 No-RAG       : {e1["Acuuracy"]:.4f}')
        print(f'    EXP_02 Naive Dense  : {e2["Acuuracy"]:.4f}')
        print(f'    EXP_03 Sparse BM25  : {e3["Acuuracy"]:.4f}')
        e4_path = REPO_ROOT / 'results' / 'exp_04_hybrid_rag__test_1273' / 'summary.json'
        if e4_path.exists():
            e4 = json.loads(e4_path.read_text())
            print(f'    EXP_04 Hybrid       : {e4["Acuuracy"]:.4f}')
        print(f'    EXP_05 Multi-Hop    : {e5_acc:.4f}')

---

**Next.** Run [`04e_exp05_ragas.ipynb`](04e_exp05_ragas.ipynb) when EXP_04 + EXP_05 baselines are both done — user is batching all 4 RAG architectures' RAGAS evaluations as a final batch.